# AI应用基础 · 工单M4.1《工业故障报告自动分类系统》 — 教师参考版
第10周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4"], check=True)
print("数据已生成: data/")

## 参考实现 `m41`（含依赖导入）

In [ ]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m41():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import make_pipeline
    import jieba

    texts, labels = [], []
    with open("data/fault_reports.csv", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            texts.append(r["text"])
            labels.append(r["label"])
    stop = set("的 了 在 是 请 尽快 处理 影响 生产 班中 巡检 发现 已 挂牌 停机".split())

    def cut(t):
        return " ".join(w for w in jieba.cut(t) if w not in stop and w.strip())

    corpus = [cut(t) for t in texts]
    n = int(len(corpus) * 0.8)
    for name, clf in [("NB", MultinomialNB()),
                      ("LR", LogisticRegression(max_iter=1000))]:
        pipe = make_pipeline(TfidfVectorizer(max_features=2000), clf)
        pipe.fit(corpus[:n], labels[:n])
        pred = pipe.predict(corpus[n:])
        print("=====", name, "=====")
        print(classification_report(labels[n:], pred))
    demo = "3号注塑机液压油温升高报警"
    pipe2 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    pipe2.fit(corpus, labels)
    print(demo, "->", pipe2.predict([cut(demo)]))


## 执行与验证

In [ ]:
m41()